# Train and publish the Covertype model

Read the prepared training and validation rows from PostgreSQL, fit a preprocessing and classification pipeline, evaluate it, then replace the stable `models/covertype/latest/model.joblib` object in MinIO. The API checks this stable object path and loads updated artifacts automatically.

In [6]:
import io
import os
import boto3
from botocore.config import Config
import joblib
import pandas as pd
import psycopg
from psycopg.rows import dict_row
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

DB_CONFIG = {
    "host": os.getenv("PROJECT_DATA_DB_HOST", "localhost"),
    "port": int(os.getenv("PROJECT_DATA_DB_PORT", "5433")),
    "dbname": os.getenv("PROJECT_DATA_DB_NAME", "cover_data"),
    "user": os.getenv("PROJECT_DATA_DB_USER", "cover_user"),
    "password": os.getenv("PROJECT_DATA_DB_PASSWORD", "cover_password"),
}
NUMERIC_FEATURES = [
    "elevation", "aspect", "slope", "horizontal_distance_to_hydrology",
    "vertical_distance_to_hydrology", "horizontal_distance_to_roadways",
    "hillshade_9am", "hillshade_noon", "hillshade_3pm",
    "horizontal_distance_to_fire_points",
]
CATEGORICAL_FEATURES = ["wilderness_area", "soil_type"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = "cover_type"


In [7]:
with psycopg.connect(**DB_CONFIG, row_factory=dict_row) as conn:
    train = pd.DataFrame(conn.execute(
        "SELECT " + ", ".join([*FEATURES, TARGET]) + " FROM cover_prepared WHERE split = \'train\'"
    ).fetchall())
    validation = pd.DataFrame(conn.execute(
        "SELECT " + ", ".join([*FEATURES, TARGET]) + " FROM cover_prepared WHERE split = \'validation\'"
    ).fetchall())
if train.empty or validation.empty:
    raise RuntimeError("Training or validation data is missing. Run 01_prepare.ipynb first.")
for frame in (train, validation):
    for column in CATEGORICAL_FEATURES:
        frame[column] = frame[column].astype(str)
print(f"Training rows: {len(train):,}; validation rows: {len(validation):,}")


Training rows: 40,670; validation rows: 8,715


## Fit and evaluate

The encoder keeps the categorical fields categorical for the histogram gradient boosting classifier. Unknown soil or wilderness codes at inference time are encoded as an explicit fallback value.

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", "passthrough", NUMERIC_FEATURES),
        ("categorical", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CATEGORICAL_FEATURES),
    ],
    verbose_feature_names_out=False,
)
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", HistGradientBoostingClassifier(
        max_iter=100,
        learning_rate=0.1,
        max_leaf_nodes=31,
        categorical_features=list(range(len(NUMERIC_FEATURES), len(FEATURES))),
        early_stopping=True,
        random_state=42,
    )),
])
model.fit(train[FEATURES], train[TARGET])
validation_predictions = model.predict(validation[FEATURES])
print(f"Validation accuracy: {accuracy_score(validation[TARGET], validation_predictions):.4f}")
print(classification_report(validation[TARGET], validation_predictions, zero_division=0))


Validation accuracy: 0.9544
              precision    recall  f1-score   support

           1       0.96      0.93      0.94      1790
           2       0.95      0.97      0.96      3230
           3       0.96      0.98      0.97      2272
           4       0.82      0.66      0.73        82
           5       0.91      0.90      0.90       284
           6       0.95      0.91      0.93       868
           7       0.97      0.95      0.96       189

    accuracy                           0.95      8715
   macro avg       0.93      0.90      0.91      8715
weighted avg       0.95      0.95      0.95      8715



## Upload the latest model to MinIO

The object key stays fixed so the API always asks for the same location. Upload happens only after training and evaluation succeed.

In [9]:
MINIO_ENDPOINT_URL = os.getenv("MINIO_ENDPOINT_URL", "http://localhost:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY", "minioadmin")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY", "minioadmin12345")
MINIO_BUCKET = os.getenv("MINIO_BUCKET", "covertype-models")
MODEL_KEY = os.getenv("MINIO_MODEL_KEY", "models/covertype/latest/model.joblib")

client = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT_URL,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name="us-east-1",
    config=Config(signature_version="s3v4", s3={"addressing_style": "path"}),
)
artifact = io.BytesIO()
joblib.dump(model, artifact)
artifact.seek(0)
client.upload_fileobj(
    artifact,
    MINIO_BUCKET,
    MODEL_KEY,
    ExtraArgs={"ContentType": "application/octet-stream"},
)
print(f"Published the evaluated model to s3://{MINIO_BUCKET}/{MODEL_KEY}")


Published the evaluated model to s3://covertype-models/models/covertype/latest/model.joblib
